# import libraries

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

In [3]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(26):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-1b-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

print(transcoder_paths)  # verify all 26 are present

{0: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-1b-pt/snapshots/b738dc06961818c011fb2e44a316352ca0f4e873/transcoder_all/layer_0_width_16k_l0_small/params.safetensors', 1: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-1b-pt/snapshots/b738dc06961818c011fb2e44a316352ca0f4e873/transcoder_all/layer_1_width_16k_l0_small/params.safetensors', 2: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-1b-pt/snapshots/b738dc06961818c011fb2e44a316352ca0f4e873/transcoder_all/layer_2_width_16k_l0_small/params.safetensors', 3: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-1b-pt/snapshots/b738dc06961818c011fb2e44a316352ca0f4e873/transcoder_all/layer_3_width_16k_l0_small/params.safetensors', 4: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-1b-pt/snapshots/b738dc06961818c011fb2e44a316352ca0f4e873/transcoder_all/layer_4

In [4]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-1b-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/circuit_tracer/transcoder/single_layer_transcoder.py:513: UserWarning: Lazy loading is not supported for GemmaScope2 format due to different key naming conventions. Setting lazy_encoder=False and lazy_decoder=False.
  warnings.warn(


In [5]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-1b-pt",
    transcoders=transcoder_set,
    backend="transformerlens",
    dtype=torch.bfloat16,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

Loaded pretrained model google/gemma-3-1b-pt into HookedTransformer


# 1. get top influential features

In [6]:
import json
import glob
from collections import defaultdict

In [7]:
GRAPH_DIR = "./graphs/gemma-3-1b"  # ← set this
DESCRIBED_LAYERS = {7, 13, 17, 22}

In [8]:
def parse_local_feat(node):
    """Extract (layer_int, local_feat_idx) from jsNodeId."""
    js = node.get("jsNodeId", "")
    try:
        layer_feat, _ = js.rsplit("-", 1)
        layer_str, feat_str = layer_feat.split("_", 1)
        return int(layer_str), int(feat_str)
    except Exception:
        return None, None

In [9]:
all_features = defaultdict(float)
 
for fpath in sorted(glob.glob(f"{GRAPH_DIR}/step-*.json")):
    with open(fpath) as f:
        data = json.load(f)
    for node in data.get("nodes", []):
        if "transcoder" not in node.get("feature_type", ""):
            continue
        layer, local_feat = parse_local_feat(node)
        if layer is None:
            continue
        inf = abs(node.get("influence", 0) or 0)
        all_features[(layer, local_feat)] += inf

### top ten from all layers

In [10]:
top_10_overall = sorted(all_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 OVERALL (all layers)")
print("=" * 70)
for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}  {in_desc}")

TOP 10 OVERALL (all layers)
 1. L25 F 4537  influence=  7.0502    undescribed
 2. L23 F 1855  influence=  6.4445    undescribed
 3. L23 F  809  influence=  6.4064    undescribed
 4. L23 F  410  influence=  6.3910    undescribed
 5. L25 F 1917  influence=  6.3811    undescribed
 6. L23 F  442  influence=  6.3649    undescribed
 7. L23 F  950  influence=  6.3216    undescribed
 8. L23 F12130  influence=  6.2611    undescribed
 9. L22 F  193  influence=  6.2044  ✓ DESCRIBED
10. L24 F14713  influence=  6.2011    undescribed


### top ten from only described layers

In [11]:
described_features = {lf: s for lf, s in all_features.items() if lf[0] in DESCRIBED_LAYERS}
top_10_described_sorted = sorted(described_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM DESCRIBED LAYERS ONLY (7, 13, 17, 22)")
print("=" * 70)
if top_10_described_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_described_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_described = [lf for lf, _ in top_10_described_sorted]

TOP 10 FROM DESCRIBED LAYERS ONLY (7, 13, 17, 22)
 1. L22 F  193  influence=  6.2044
 2. L22 F 2363  influence=  5.9199
 3. L22 F  572  influence=  5.8578
 4. L22 F12043  influence=  5.5238
 5. L22 F 1756  influence=  5.3414
 6. L22 F  336  influence=  5.2388
 7. L22 F   18  influence=  5.1310
 8. L22 F 1900  influence=  5.1262
 9. L22 F 1167  influence=  4.9080
10. L22 F 4644  influence=  4.8542


### top ten from every other layer

In [12]:
other_features = {lf: s for lf, s in all_features.items() if lf[0] not in DESCRIBED_LAYERS}
top_10_undescribed_sorted = sorted(other_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM OTHER LAYERS (not 7, 13, 17, 22)")
print("=" * 70)
if top_10_undescribed_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_undescribed_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_undescribed = [lf for lf, _ in top_10_undescribed_sorted]

TOP 10 FROM OTHER LAYERS (not 7, 13, 17, 22)
 1. L25 F 4537  influence=  7.0502
 2. L23 F 1855  influence=  6.4445
 3. L23 F  809  influence=  6.4064
 4. L23 F  410  influence=  6.3910
 5. L25 F 1917  influence=  6.3811
 6. L23 F  442  influence=  6.3649
 7. L23 F  950  influence=  6.3216
 8. L23 F12130  influence=  6.2611
 9. L24 F14713  influence=  6.2011
10. L23 F 2922  influence=  6.1814


### summary

In [13]:
print(f"Total unique transcoder features: {len(all_features)}")
print(f"Features in described layers:     {len(described_features)}")
print(f"Features in other layers:         {len(other_features)}")

Total unique transcoder features: 2448
Features in described layers:     346
Features in other layers:         2102


# intervention

### all 10 features suppressed

In [14]:
# Your prompt
prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

# Target position (typically -1 for last token, or specify the rhyme token position)
TARGET_POS = -1

In [15]:
features_to_intervene = [lf for lf, _ in top_10_overall]
intervention_tuples = [(layer, TARGET_POS, feat, 0.0) for layer, feat in features_to_intervene]

print("\n" + "=" * 70)
print("INTERVENTION TUPLES (top 10 features, suppressed at target position)")
print("=" * 70)
for i, (layer, pos, feat, val) in enumerate(intervention_tuples, 1):
    print(f"{i:2d}. Layer {layer:2d}, Position {pos:3d}, Feature {feat:5d}, Value {val}")


INTERVENTION TUPLES (top 10 features, suppressed at target position)
 1. Layer 25, Position  -1, Feature  4537, Value 0.0
 2. Layer 23, Position  -1, Feature  1855, Value 0.0
 3. Layer 23, Position  -1, Feature   809, Value 0.0
 4. Layer 23, Position  -1, Feature   410, Value 0.0
 5. Layer 25, Position  -1, Feature  1917, Value 0.0
 6. Layer 23, Position  -1, Feature   442, Value 0.0
 7. Layer 23, Position  -1, Feature   950, Value 0.0
 8. Layer 23, Position  -1, Feature 12130, Value 0.0
 9. Layer 22, Position  -1, Feature   193, Value 0.0
10. Layer 24, Position  -1, Feature 14713, Value 0.0


In [16]:
print("\n" + "=" * 70)
print("GENERATING WITH ORIGINAL FEATURES (no intervention)")
print("=" * 70)

pre_intervention_text = model.feature_intervention_generate(
    prompt,
    [],  # empty list = no interventions
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{pre_intervention_text}")

# ─────────────────────────────────────────────────────────────
# Generate with all 20 features suppressed
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("GENERATING WITH ALL 10 FEATURES SUPPRESSED")
print("=" * 70)

post_intervention_text = model.feature_intervention_generate(
    prompt,
    intervention_tuples,
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{post_intervention_text}")


GENERATING WITH ORIGINAL FEATURES (no intervention)


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
He saw a carrot and had to grab it,

GENERATING WITH ALL 10 FEATURES SUPPRESSED


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
 mmm, mmm, mmm, m


In [17]:
print("\n" + "=" * 70)
print("COMPARISON: PRE vs POST INTERVENTION")
print("=" * 70)

from circuit_tracer.utils.demo_utils import display_generations_comparison

display_generations_comparison(
    prompt,
    [pre_intervention_text],
    [post_intervention_text]
)


COMPARISON: PRE vs POST INTERVENTION


### get pseudo clerp description

In [18]:
def pseudo_clerp_topk(model, layer, local_feat, tokenizer, top_k=10):
    """
    Project feature decoder direction through unembedding matrix.
    Returns top-k predicted tokens for this feature.
    """
    tc = model.transcoders[layer]
    # W_dec is the direction in residual space the feature 'writes' to
    W_dec = tc.W_dec[local_feat] # shape: [d_model]
    
    with torch.no_grad():
        # model.unembed.W_U shape is usually [d_model, vocab_size]
        W_U = model.unembed.W_U 
        
        # We want to project W_dec (d_model) onto each vocab entry (d_model)
        # So we do: [vocab_size, d_model] @ [d_model]
        # We use W_U.T to get [vocab_size, d_model]
        logits = torch.matmul(W_U.T, W_dec.to(W_U.dtype)) # result: [vocab_size]
    
    # Get top-k tokens
    top_ids = logits.topk(top_k).indices.tolist()
    top_tokens = [tokenizer.decode([i]) for i in top_ids]
    
    return top_tokens

# ─────────────────────────────────────────────────────────────
# Run pseudo-clerp for all top 10 overall features
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print("PSEUDO-CLERP FOR TOP 10 OVERALL FEATURES")
print("=" * 70)

for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    tokens = pseudo_clerp_topk(model, layer, feat, tokenizer, top_k=10)
    
    # Check if this is a described feature
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    
    print(f"\n{i:2d}. L{layer:2d} F{feat:5d} (influence={score:8.4f}) {in_desc}")
    print(f"    Top tokens: {tokens}")

PSEUDO-CLERP FOR TOP 10 OVERALL FEATURES

 1. L25 F 4537 (influence=  7.0502)   undescribed
    Top tokens: ['itabbo', 'iftoire', 'itabbam', ' Margarita', ' Đình', 'butterfly', 'abhavena', 'ল্যান্ড', ' Padukone', 'agamanam']

 2. L23 F 1855 (influence=  6.4445)   undescribed
    Top tokens: ['عيه', 'ONSE', ' bö', ' bande', '文章', 'ब्रे', ' dp', ' عنہ', 'عي', '颗粒']

 3. L23 F  809 (influence=  6.4064)   undescribed
    Top tokens: ['电压', 'ிருக்கிற', ')});', 'transferase', ' ద', ' Cyclic', ' miz', 'គ្រប់', "'});", ' dél']

 4. L23 F  410 (influence=  6.3910)   undescribed
    Top tokens: ['ivil', 'న్నా', 'oni', 'ặng', '朕', 'ns', 'ift', ' coli', 'apped', ' राजेंद्र']

 5. L25 F 1917 (influence=  6.3811)   undescribed
    Top tokens: ['<unused62>', '<unused341>', '<unused960>', '<unused1244>', ' Ary', ' Doppler', ' Aspergillus', '<unused1221>', '<unused1829>', '<unused357>']

 6. L23 F  442 (influence=  6.3649)   undescribed
    Top tokens: ['�', 'uffles', ' কবি', '网红', 'uomo', 'рова', ' за

In [19]:
# Pseudo-clerp descriptions for top 10 overall features from Gemini 3 (fast)
pseudo_clerps = {
    (25, 4537): "Multilingual word fragments, proper names, and biological terms.",
    (23, 1855): "Diverse multilingual scripts including Arabic, Chinese, and Hindi.",
    (23, 809): "Code syntax, chemical suffixes, and non-Latin character strings.",
    (23, 410): "Morphological suffixes, Asian scripts, and biological genus names.",
    (25, 1917): "Unused model tokens alongside scientific and medical terminology.",
    (23, 442): "Encoding artifacts, multilingual fragments, and variants of 'shuffle'.",
    (23, 950): "Unused placeholder tokens and South Asian script components.",
    (23, 12130): "High-frequency sentence starters, pronouns, and demonstrative determiners.",
    (22, 193): "Financial terms, security concepts, and multilingual programming nouns.",
    (24, 14713): "Uppercase industrial acronyms and specific English proper surnames.",
}

### one by one suppression

In [20]:
print("Suppressing one feature at a time to isolate effects...")

ablation_results = []

for layer, feat in features_to_intervene:  # all 10 features
    single_intervention = [(layer, TARGET_POS, feat, 0.0)]
    
    try:
        ablated_text = model.feature_intervention_generate(
            prompt,
            single_intervention,
            do_sample=False,
            verbose=False
        )[0]
        
        # Check if output changed
        changed = ablated_text != pre_intervention_text
        status = "CHANGED" if changed else "unchanged"
        
        # Get clerp (try pseudo_clerps first, fall back to clerps)
        clerp_desc = pseudo_clerps.get((layer, feat)) or clerps.get((layer, feat))
        clerp_str = f" | {clerp_desc}" if clerp_desc else ""
        
        ablation_results.append({
            'layer': layer,
            'feat': feat,
            'changed': changed,
            'text': ablated_text,
            'clerp': clerp_desc
        })
        
        print(f"\nL{layer:2d} F{feat:5d}: {status}{clerp_str}")
        if changed:
            print(f"  → {ablated_text}")
    
    except Exception as e:
        print(f"L{layer:2d} F{feat:5d}: ERROR ({str(e)[:50]})")

Suppressing one feature at a time to isolate effects...

L25 F 4537: CHANGED | Multilingual word fragments, proper names, and biological terms.
  → A rhyming couplet:
He saw a carrot and had to grab it,
 mmm, I love carrots.

A rh

L23 F 1855: CHANGED | Diverse multilingual scripts including Arabic, Chinese, and Hindi.
  → A rhyming couplet:
He saw a carrot and had to grab it,
greased his teeth and ate it.

A

L23 F  809: CHANGED | Code syntax, chemical suffixes, and non-Latin character strings.
  → A rhyming couplet:
He saw a carrot and had to grab it,
 ослабел, и упал.



L23 F  410: CHANGED | Morphological suffixes, Asian scripts, and biological genus names.
  → A rhyming couplet:
He saw a carrot and had to grab it,
دللته و أكلته و أكل

L25 F 1917: CHANGED | Unused model tokens alongside scientific and medical terminology.
  → A rhyming couplet:
He saw a carrot and had to grab it,
 рабби, рабби, раб

L23 F  442: CHANGED | Encoding artifacts, multilingual fragments, and variants of '

# using clerp descriptions for SAE

### get top 10 from described layers

In [21]:
described_features = {lf: s for lf, s in all_features.items() if lf[0] in DESCRIBED_LAYERS}
top_10_described_sorted = sorted(described_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM DESCRIBED LAYERS ONLY (7, 13, 17, 22)")
print("=" * 70)
if top_10_described_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_described_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_described = [lf for lf, _ in top_10_described_sorted]

TOP 10 FROM DESCRIBED LAYERS ONLY (7, 13, 17, 22)
 1. L22 F  193  influence=  6.2044
 2. L22 F 2363  influence=  5.9199
 3. L22 F  572  influence=  5.8578
 4. L22 F12043  influence=  5.5238
 5. L22 F 1756  influence=  5.3414
 6. L22 F  336  influence=  5.2388
 7. L22 F   18  influence=  5.1310
 8. L22 F 1900  influence=  5.1262
 9. L22 F 1167  influence=  4.9080
10. L22 F 4644  influence=  4.8542


### get clerp descriptions from neuronpedia

In [22]:
import requests
import time

def fetch_clerps_batch(features_list, model_id="gemma-3-1b", slug_suffix="gemmascope-2-res-16k", delay=0.3):
    """Fetch clerp descriptions for a list of (layer, feature_idx) tuples."""
    clerps = {}
    
    for i, (layer, feat) in enumerate(features_list):
        source = f"{layer}-{slug_suffix}"
        url = f"https://www.neuronpedia.org/api/feature/{model_id}/{source}/{feat}"
        
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                exps = r.json().get("explanations", [])
                desc = exps[0].get("description") if exps else None
                clerps[(layer, feat)] = desc
                status = "✓" if desc else "✗ (no desc)"
            else:
                clerps[(layer, feat)] = None
                status = f"✗ (HTTP {r.status_code})"
        except Exception as e:
            clerps[(layer, feat)] = None
            status = f"✗ ({str(e)[:30]})"
        
        print(f"  [{i+1}/{len(features_list)}] L{layer:2d} F{feat:5d}: {status}")
        time.sleep(delay)
    
    return clerps

# Extract the top 10 described features
features_to_fetch = [lf for lf, _ in top_10_described_sorted]

# Fetch clerps
print("Fetching clerps for top 10 described features...")
clerps = fetch_clerps_batch(features_to_fetch)

# Print results
print("\n" + "="*70)
print("TOP 10 DESCRIBED FEATURES WITH CLERPS")
print("="*70)
for (layer, feat), score in top_10_described_sorted:
    desc = clerps.get((layer, feat))
    print(f"L{layer:2d} F{feat:5d} (influence={score:8.4f}): {desc or '[no description found]'}")

Fetching clerps for top 10 described features...
  [1/10] L22 F  193: ✓
  [2/10] L22 F 2363: ✓
  [3/10] L22 F  572: ✓
  [4/10] L22 F12043: ✓
  [5/10] L22 F 1756: ✓
  [6/10] L22 F  336: ✓
  [7/10] L22 F   18: ✓
  [8/10] L22 F 1900: ✓
  [9/10] L22 F 1167: ✓
  [10/10] L22 F 4644: ✓

TOP 10 DESCRIBED FEATURES WITH CLERPS
L22 F  193 (influence=  6.2044): wavelengths
L22 F 2363 (influence=  5.9199): Okay, let's
L22 F  572 (influence=  5.8578): programming explanations
L22 F12043 (influence=  5.5238): requests to show something
L22 F 1756 (influence=  5.3414): gunshot
L22 F  336 (influence=  5.2388): would you like me to
L22 F   18 (influence=  5.1310): Buddhist spiritual traditions
L22 F 1900 (influence=  5.1262): for each
L22 F 1167 (influence=  4.9080): describing someone sexually
L22 F 4644 (influence=  4.8542): press Enter


### feature suppression one at a time (top five)

In [23]:
print("Suppressing one feature at a time to isolate effects...")

all_features = top_10_described
ablation_results = []

for layer, feat in all_features[:5]:  # just first 5 for speed
    single_intervention = [(layer, TARGET_POS, feat, 0.0)]
    
    try:
        ablated_text = model.feature_intervention_generate(
            prompt,
            single_intervention,
            do_sample=False,
            verbose=False
        )[0]
        
        # Check if output changed
        changed = ablated_text != pre_intervention_text
        status = "CHANGED" if changed else "unchanged"
        
        # Get clerp if available
        clerp_desc = clerps.get((layer, feat)) if (layer, feat) in clerps else None
        clerp_str = f" | {clerp_desc}" if clerp_desc else ""
        
        ablation_results.append({
            'layer': layer,
            'feat': feat,
            'changed': changed,
            'text': ablated_text,
            'clerp': clerp_desc
        })
        
        print(f"\nL{layer:2d} F{feat:5d}: {status}{clerp_str}")
        if changed:
            print(f"  → {ablated_text}")
    
    except Exception as e:
        print(f"L{layer:2d} F{feat:5d}: ERROR ({str(e)[:50]})")

Suppressing one feature at a time to isolate effects...

L22 F  193: CHANGED | wavelengths
  → A rhyming couplet:
He saw a carrot and had to grab it,
 Oldham, Lancashire, England.

A rhyming

L22 F 2363: CHANGED | Okay, let's
  → A rhyming couplet:
He saw a carrot and had to grab it,
 dolomites,
 and he had to go.

L22 F  572: CHANGED | programming explanations
  → A rhyming couplet:
He saw a carrot and had to grab it,
<unused623> ने एक कद्दू देखा और उसे पकड़

L22 F12043: CHANGED | requests to show something
  → A rhyming couplet:
He saw a carrot and had to grab it,
…
He saw a carrot and had to grab

L22 F 1756: CHANGED | gunshot
  → A rhyming couplet:
He saw a carrot and had to grab it,
 wprowadzenie do angielskiego
and he had to
